# ============================================================
# 1. IMPORTS
# ============================================================


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV

# ============================================================
# 2. LOAD & BASIC CLEANING
# ============================================================

In [2]:
flight_df = pd.read_csv(r"raw_bangladesh_data\Flight_Price_Dataset_of_Bangladesh.csv")

df = flight_df.copy()

# Drop irrelevant & leakage columns
df = df.drop(columns=[
    "Source Name",
    "Destination Name",
    "Base Fare (BDT)",
    "Tax & Surcharge (BDT)"
])

# Convert datetime
df["Departure Date & Time"] = pd.to_datetime(df["Departure Date & Time"])
df["Arrival Date & Time"] = pd.to_datetime(df["Arrival Date & Time"])


# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

In [3]:
df["Month"] = df["Departure Date & Time"].dt.month
df["Day"] = df["Departure Date & Time"].dt.day
df["Weekday"] = df["Departure Date & Time"].dt.weekday
df["Hour"] = df["Departure Date & Time"].dt.hour
df["Is_Weekend"] = df["Weekday"].isin([5,6]).astype(int)


df["Stopovers"] = df["Stopovers"].replace({"Direct": 0})

df["Stopovers"] = df["Stopovers"].str.extract(r'(\d+)')
df["Stopovers"] = df["Stopovers"].fillna(0).astype(int)




df = df.drop(columns=[
    "Departure Date & Time",
    "Arrival Date & Time"
])

# ============================================================
# 4. DEFINE FEATURES & TARGET
# ============================================================

In [4]:
X = df.drop("Total Fare (BDT)", axis=1)
y = df["Total Fare (BDT)"]

numeric_features = [
    "Duration (hrs)",
    "Days Before Departure",
    "Month",
    "Day",
    "Weekday",
    "Hour",
    "Is_Weekend",
    "Stopovers"
]

categorical_features = [
    "Airline",
    "Source",
    "Destination",
    "Aircraft Type",
    "Class",
    "Booking Source",
    "Seasonality"
]

# ============================================================
# 5. PREPROCESSING PIPELINE
# ============================================================

In [5]:
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [14]:
# Linear Regression Pipeline
lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])
y = np.log1p(df["Total Fare (BDT)"])


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr_pipeline.fit(X_train, y_train)

y_pred = lr_pipeline.predict(X_test)

# Convert back
y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test)

print("R2:", r2_score(y_test_actual, y_pred_actual))
print("MAE:", mean_absolute_error(y_test_actual, y_pred_actual))
print("RMSE:", np.sqrt(mean_squared_error(y_test_actual, y_pred_actual)))


R2: 0.6504649565160361
MAE: 28533.190947478313
RMSE: 48271.29792322513


In [15]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=50,      # smaller
        max_depth=10,         # limit tree depth
        random_state=42,
        n_jobs=-1             # parallel trees
    ))
])


#### Hyperparameter Tuning

In [ ]:
param_dist = {
    "model__n_estimators": [50, 100],
    "model__max_depth": [10, 20]
}

random_search = RandomizedSearchCV(
    rf_pipeline,
    param_dist,
    n_iter=3,        # try only 3 combos
    cv=3,            # reduce folds
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

print(X_train.shape)



In [12]:
print("Average Fare:", y.mean())


Average Fare: 71030.31619864231


In [13]:
print(y.describe())


count     57000.000000
mean      71030.316199
std       81769.199536
min        1800.975688
25%        9602.699787
50%       41307.544990
75%      103800.906963
max      558987.332444
Name: Total Fare (BDT), dtype: float64
